# What is actually inside ChatGPT? 🤖

Completely different tool, completely different job. Let's take one apart.

We'll build a single neuron by hand, then run a **real** language model — the
same kind of thing behind ChatGPT, just small enough to run on this free
computer.

In [ ]:
# --- setup: run me first! ---
import pandas as pd, numpy as np, matplotlib.pyplot as plt

DATA = "https://raw.githubusercontent.com/emcramer/psi-class-2026/main/data/processed"

plt.rcParams.update({"figure.figsize": (8, 5.5), "font.size": 13,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e1e0d9",
                     "figure.facecolor": "white", "axes.facecolor": "white"})
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"
print("ready")

## Part 1 — One neuron, built from scratch

A neuron does three things: **multiply** each input by a weight, **add** them
up, and **squash** the answer to a number between 0 and 1.

That's it. That's the whole thing. Let's teach one to spot immune cells.

In [ ]:
cells = pd.read_csv(f"{DATA}/cells_sample.csv")

# two inputs, one answer
X = cells[["CD45", "Pan-Keratin"]].values
y = (~cells.celltype.isin(["Tumor", "Endothelial", "Mesenchymal"])).astype(float).values

X = (X - X.mean(0)) / X.std(0)      # scale
w = np.zeros(2); b = 0.0            # the neuron starts knowing nothing

def squash(z):
    return 1 / (1 + np.exp(-z))

for step in range(300):
    guess = squash(X @ w + b)       # multiply, add, squash
    error = guess - y               # how wrong were we?
    w -= 0.5 * (X.T @ error) / len(y)   # nudge the weights
    b -= 0.5 * error.mean()
    if step % 100 == 0:
        print(f"step {step:3d}  accuracy {((guess > 0.5) == y).mean():.1%}")

print(f"\nfinal accuracy: {((squash(X @ w + b) > 0.5) == y).mean():.1%}")
print(f"learned weights: CD45 {w[0]:+.2f}, Pan-Keratin {w[1]:+.2f}")

It taught itself that **high CD45 means immune** (positive weight) and **high
keratin means not immune** (negative weight) — which is exactly what a
biologist would tell you.

A language model is this same neuron, roughly **a hundred billion times over**.

---
## Part 2 — Words have to become numbers first

A model can't read letters. Every piece of text gets chopped into **tokens**
and each token becomes a number.

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

for text in ["Machine learning", "unbelievable", "1847362", "biology"]:
    pieces = [tokenizer.decode([i]) for i in tokenizer(text).input_ids]
    print(f"{text:20s} -> {pieces}")

### 🔍 Your turn

Put **your own name** in the cell below. Does it survive in one piece, or get
chopped up? Try a friend's name, a slang word, an emoji.

*(This is genuinely why models are bad at spelling and counting letters — they
never see letters.)*

In [ ]:
my_text = "put your name here"

pieces = [tokenizer.decode([i]) for i in tokenizer(my_text).input_ids]
print(pieces)
print(f"{len(pieces)} tokens")

---
## Part 3 — A real language model

Now we download **GPT-2** — a real model from 2019. When it came out, OpenAI
said it was too dangerous to release publicly.

It's about 500 MB, and it downloads to Google's computer, not yours.

In [ ]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()
print(f"{sum(p.numel() for p in model.parameters()):,} numbers inside this model")

### All it does is guess the next word.

That's the entire trick. Give it some text, it produces a probability for every
possible next token. Let's watch.

In [ ]:
prompt = "The patient was diagnosed with breast"

ids = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    logits = model(**ids).logits[0, -1]
probs = torch.softmax(logits, dim=-1)

top = torch.topk(probs, 8)
words = [tokenizer.decode(i).strip() for i in top.indices]

plt.barh(range(8)[::-1], top.values.numpy(), color=BLUE)
plt.yticks(range(8)[::-1], words); plt.xlabel("probability")
plt.title(f'"{prompt} ___"'); plt.grid(False)
plt.show()

It is **94% sure** the next word is "cancer".

To write a sentence, it picks one, sticks it on the end, and asks the same
question again. And again. That's all "writing" is, for a language model.

### 🔍 Your turn — try your own prompt

In [ ]:
my_prompt = "The best thing about high school is"

ids = tokenizer(my_prompt, return_tensors="pt")
with torch.no_grad():
    probs = torch.softmax(model(**ids).logits[0, -1], dim=-1)
top = torch.topk(probs, 8)
for p, i in zip(top.values, top.indices):
    print(f"{p:6.1%}  {tokenizer.decode(i)!r}")

---
## Part 4 — The creativity dial

If it always picked the single most likely word it would be boring and
repetitive. So we let it roll dice — and **temperature** decides how loaded
those dice are.

Here's the key thing: temperature does **not** cross any words off the list. It
takes the *same* list and stretches or squashes the odds:

> `temperature < 1` → gaps get bigger → the favorite wins even more often
> `temperature > 1` → gaps get smaller → long shots get a real chance

Watch the same seven words get re-weighted.

In [ ]:
ids = tokenizer("She opened the door and saw a", return_tensors="pt")
with torch.no_grad():
    scores = model(**ids).logits[0, -1]      # raw scores, before probabilities

top7 = torch.topk(torch.softmax(scores, -1), 7).indices
words = [tokenizer.decode(i).strip() for i in top7]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, temp in zip(axes, [0.5, 1.0, 2.0]):
    p = torch.softmax(scores / temp, -1)[top7]    # <-- divide by temperature
    p = (p / p.sum()).numpy()                     # share among these seven
    ax.barh(range(7)[::-1], p, color=BLUE)
    ax.set_yticks(range(7)[::-1]); ax.set_yticklabels(words)
    ax.set_title(f"temperature {temp}"); ax.set_xlim(0, 1); ax.grid(False)
plt.tight_layout(); plt.show()

Same seven words every time. Only the odds changed.

*(The knob that genuinely does delete words is a different one, called
**top-k** or **top-p** — it keeps only the best few and throws the rest away.)*

---
## Part 5 — A model that can actually talk

GPT-2 is from 2019 and it rambles. Here's a model from 2024 that's only 4×
bigger but was trained far better — small enough to still run right here.

In [ ]:
chat_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
chat_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
print("loaded")

In [ ]:
question = "In one sentence, what is a T cell?"

text = chat_tok.apply_chat_template([{"role": "user", "content": question}],
                                    tokenize=False, add_generation_prompt=True)
ids = chat_tok(text, return_tensors="pt")
out = chat_model.generate(**ids, max_new_tokens=80, do_sample=False)
print(chat_tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True))

### 🔍 Your turn — ask it anything

Change the question above and run it again. Try to catch it being **confidently
wrong** — ask about something you know really well, or something very recent.

That's the most important thing to understand about these tools: it is picking
likely next words. It is not looking anything up, and it does not know when
it's wrong.

---
## Part 6 — Now turn the dial on a model that can talk

Same temperature setting as Part 4, but on a model good enough that you can
*hear* the difference.

In [ ]:
question = "In one sentence, why is the sky blue?"
text = chat_tok.apply_chat_template([{"role": "user", "content": question}],
                                    tokenize=False, add_generation_prompt=True)
ids = chat_tok(text, return_tensors="pt")

for temp in [0.3, 1.0, 1.8]:
    torch.manual_seed(3)
    out = chat_model.generate(**ids, max_new_tokens=45, do_sample=True,
                              temperature=temp, top_k=0, top_p=1.0)
    print(f"--- temperature {temp} ---")
    print(chat_tok.decode(out[0][ids.input_ids.shape[1]:],
                          skip_special_tokens=True).strip(), "\n")

### ⚠️ Read the first two answers carefully.

Both of them sound completely confident. **Both of them are wrong.**

The sky is blue because air scatters short (blue) wavelengths much more than
long ones — *Rayleigh scattering*. It is not about reflection, and it is
certainly not "white or gray."

Nothing in the model flagged those answers as wrong, because nothing in the
model was ever checking. It was picking likely next words, and a confident
sentence is a *likely* sentence.

That's the single most useful thing to know about these tools.

---
## The scale ladder

| model | numbers inside | year |
|---|---|---|
| the neuron you built | **3** | today |
| GPT-2 | 124,000,000 | 2019 |
| the one you just chatted with | 500,000,000 | 2024 |
| ChatGPT / Claude | ~1,000,000,000,000 | now |

Same three operations — multiply, add, squash — all the way up.

---
### 🚀 If you have time

- Ask the chat model a math question. Then a harder one.
- Ask it about a topic you're an expert in and grade its answer.
- Set `do_sample=True, temperature=1.5` on the chat model and watch it lose it.